# 🧪 Tutorial: Local RAG System for Safety Data Sheets (SDS)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

*Based on the Chemical Engineering Tutorial by **Dr. Gloria M. Monsalve-Bravo** (School of Chemical Engineering, The University of Queensland).*

---

### 📖 Introduction & System Overview
In chemical plants, laboratories, and engineering facilities, **Safety Data Sheets (SDS)** provide vital information on chemical identity, hazard classifications, exposure limits, personal protective equipment (PPE), handling instructions, and emergency first aid procedures.

However, finding specific safety guidance quickly across hundreds of pages of technical documentation can be challenging. A standard Large Language Model (LLM) might hallucinate safety advice or confuse different chemicals. 

This notebook implements a complete **Retrieval-Augmented Generation (RAG)** pipeline designed to provide **traceable, grounded, and verified answers** directly from your Safety Data Sheets:
1. **Interactive File Ingestion**: Upload any PDF documents directly from your computer (no Google Drive mounting required).
2. **Metadata Tracking**: Extract and preserve page numbers and document sources for auditability.
3. **Recursive Chunking**: Split multi-page documents into semantically coherent segments (`chunk_size=1000`, `chunk_overlap=100`).
4. **Dense Vector Embeddings**: Convert chunks into normalized semantic vectors using `BAAI/bge-small-en` on CPU.
5. **Vector Database**: Store and index vectors using **ChromaDB**.
6. **Cosine Distance Filtering**: Apply a strict similarity threshold ($0.60$) to reject out-of-scope questions and eliminate hallucinations.
7. **Strict System Prompt**: Enforce mandatory page citations and restrict the model strictly to the provided context.
8. **Local Language Model**: Run **Llama 3.2 (3B)** locally inside the Google Colab environment via **Ollama**, ensuring complete data privacy and zero API costs.

```
┌─────────────────────────────────┐
│  Upload Files (Any PDFs / SDS)  │
└────────────────┬────────────────┘
                 │
                 ▼
┌─────────────────────────────────┐
│ Recursive Chunking (1000 / 100) │
└────────────────┬────────────────┘
                 │
                 ▼
┌─────────────────────────────────┐
│ BGE Embeddings & ChromaDB Index │
└────────────────┬────────────────┘
                 │
   User Query    ▼
  ───────────> ┌─────────────────────────────────┐
               │ Semantic Search (Top k=3)       │
               └────────────────┬────────────────┘
                                │
                                ▼
               ┌─────────────────────────────────┐
               │ Cosine Distance Threshold Check │
               │ (all chunks > 0.60 => Reject)   │
               └────────────────┬────────────────┘
                                │
                                ▼
               ┌─────────────────────────────────┐
               │ Grounded Prompt + REF Pages     │
               └────────────────┬────────────────┘
                                │
                                ▼
               ┌─────────────────────────────────┐
               │ Local Llama 3.2 (via Ollama)    │
               └────────────────┬────────────────┘
                                │
                                ▼
               ┌─────────────────────────────────┐
               │ Verified Answer with Page Refs  │
               └─────────────────────────────────┘
```

> **⚡ Hardware Acceleration Notice:** You can switch Colab's runtime to GPU (**Runtime** → **Change runtime type** → **T4 GPU**) for faster Ollama LLM inference. The lightweight embedding model runs efficiently on CPU.


## Step 1: Install Dependencies
We install the required open-source libraries:
- `langchain`, `langchain-core`, `langchain-community`: Core RAG framework and abstractions.
- `langchain-text-splitters`: Recursive character chunking.
- `langchain-huggingface`: HuggingFace embeddings model integration.
- `langchain-ollama`: Connection to local Ollama inference service.
- `langchain-chroma`, `chromadb`: Persistent vector database.
- `transformers`, `sentence-transformers`: Transformer architectures and tokenizers.
- `pypdf`: PDF text and metadata extraction.
- `reportlab`: Programmatic PDF generation for sample datasets.
- `ipywidgets`: Rich interactive UI elements.


In [ ]:
# Install required libraries
!pip install -q \
    langchain \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-ollama \
    langchain-chroma \
    chromadb \
    transformers \
    sentence-transformers \
    pypdf \
    reportlab \
    ipywidgets

print("✅ All Python dependencies installed successfully!")


## Step 2: Install & Launch Ollama in Google Colab
**Ollama** enables running local LLMs (such as Meta's **Llama 3.2**) directly inside the Colab virtual machine. This means:
- **Zero API costs**: No OpenAI or Anthropic API keys required.
- **Data Privacy**: Your documents never leave the Colab instance.
- **Full Control**: Hardware acceleration on CPU or T4 GPU.

We will:
1. Download and install the Ollama Linux binary.
2. Launch the Ollama daemon in the background (`ollama serve`).
3. Download the `llama3.2:3b` model (or `llama3.2:1b` if preferred).


In [ ]:
# Install zstd and pciutils (GPU detection), then install Ollama
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess
import time
import urllib.request

# Check if Ollama server is already active
server_ready = False
try:
    with urllib.request.urlopen("http://127.0.0.1:11434") as resp:
        if resp.status == 200:
            print("✅ Ollama server is already running on http://127.0.0.1:11434!")
            server_ready = True
except Exception:
    pass

# If not running, start it in the background
if not server_ready:
    print("🚀 Launching Ollama daemon in the background...")
    ollama_process = subprocess.Popen(["ollama", "serve"])
    time.sleep(4)

    for attempt in range(1, 11):
        try:
            with urllib.request.urlopen("http://127.0.0.1:11434") as resp:
                if resp.status == 200:
                    print(f"✅ Ollama server is up and listening on http://127.0.0.1:11434 (attempt {attempt})")
                    server_ready = True
                    break
        except Exception:
            time.sleep(1)

if not server_ready:
    print("⚠️ Ollama status check timed out; proceeding to model download.")


In [ ]:
#@title 🤖 Select and Download Language Model
#@markdown The tutorial uses `llama3.2:3b` (2.0 GB). You can also choose `llama3.2:1b` (1.3 GB) for a lighter/faster download.
MODEL_NAME = "llama3.2:3b" #@param ["llama3.2:3b", "llama3.2:1b", "mistral", "phi3:mini"]

print(f"📥 Pulling model '{MODEL_NAME}' into Ollama (takes ~1-2 minutes)...")
!ollama pull {MODEL_NAME}
print(f"✅ Model '{MODEL_NAME}' downloaded and ready for inference!")


## Step 3: Educational Foundations — Tokenization & Embeddings
Before building the full RAG pipeline, let's understand how text is transformed into dense vector representations:
1. **Tokenization**: Breaking raw words into subword tokens and numeric token IDs (`Word_to_Embeddings_Example.py`).
2. **Dense Embeddings**: Transforming text into 384-dimensional semantic vectors (`BAAI/bge-small-en`).
3. **Cosine Similarity**: Computing the dot product of normalized vectors to measure semantic relatedness between chemical concepts (`Embeddings_Similarity_Example.py`).


In [ ]:
# ---------------------------------------------------
# STEP 3A: Tokenization and Embedding Dimensions
# ---------------------------------------------------
import numpy as np
from transformers import AutoTokenizer
from langchain_huggingface import HuggingFaceEmbeddings

preview_model_name = "BAAI/bge-small-en"
tokenizer = AutoTokenizer.from_pretrained(preview_model_name)

emb_preview = HuggingFaceEmbeddings(
    model_name=preview_model_name,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

sample_word = "reactor"
tokens = tokenizer.tokenize(sample_word)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
sample_vector = emb_preview.embed_query(sample_word)

print(f"🔤 Input word: '{sample_word}'")
print(f"🧩 Subword Tokens: {tokens}")
print(f"🔢 Token IDs: {token_ids}")
print(f"📐 Embedding Dimension: {len(sample_vector)} floating-point numbers")
print(f"📊 First 5 vector components: {[round(x, 4) for x in sample_vector[:5]]}")

# ---------------------------------------------------
# STEP 3B: Cosine Similarity between Concepts
# ---------------------------------------------------
v_methanol = emb_preview.embed_query("Methanol")
v_ethanol = emb_preview.embed_query("Ethanol")
v_basketball = emb_preview.embed_query("Basketball")

def cosine_similarity(a, b):
    # Normalized embeddings: dot product equals cosine similarity
    return float(np.dot(a, b))

print("\n=== COSINE SIMILARITY COMPARISON ===")
print(f"Methanol vs Methanol (identical):   {cosine_similarity(v_methanol, v_methanol):.4f}")
print(f"Methanol vs Ethanol (related):      {cosine_similarity(v_methanol, v_ethanol):.4f}")
print(f"Methanol vs Basketball (unrelated): {cosine_similarity(v_methanol, v_basketball):.4f}")


## Step 4: Interactive File Upload Section
> **📂 Upload Your Documents (PDFs)**
> 
> You can upload whatever files you want directly from your computer (e.g. Safety Data Sheets, equipment manuals, technical documents).
>
> Run the code cell below and click the **"Choose Files"** button.


In [ ]:
#@title 📤 Upload Files (PDFs)
import os
from google.colab import files

DATA_PATH = "docs"
os.makedirs(DATA_PATH, exist_ok=True)

# Upload files interactively
print("Click 'Choose Files' below to upload your documents:")
uploaded = files.upload()

# Save uploaded files to the docs/ directory
for filename, content in uploaded.items():
    file_path = os.path.join(DATA_PATH, filename)
    with open(file_path, "wb") as f:
        f.write(content)
    print(f"✅ Saved: {filename}")

# Show current files
print(f"\n📁 Current files in docs/: {os.listdir(DATA_PATH)}")


## Step 5: Document Loading & Metadata Extraction
We load documents page-by-page using `PyPDFLoader` (or `TextLoader` for text files).
For each page, we track:
- `source`: The file path
- `page`: 0-indexed page number (which will be converted to 1-indexed for citations)

This metadata is essential so that when an answer is retrieved, the user can verify the exact document and page number.


In [ ]:
import time
from langchain_community.document_loaders import PyPDFLoader, TextLoader

print("📄 Ingesting document(s) from knowledge base...")
start_time = time.time()
documents = []

file_list = [f for f in sorted(os.listdir(DATA_PATH)) if not f.startswith('.')]

if not file_list:
    raise ValueError(f"No files found in '{DATA_PATH}/'! Please upload files in Step 4.")

for filename in file_list:
    file_path = os.path.join(DATA_PATH, filename)
    print(f"📥 Loading: {filename}")

    if filename.lower().endswith(".pdf"):
        loader = PyPDFLoader(file_path)
        docs = loader.load()
    elif filename.lower().endswith((".txt", ".md")):
        loader = TextLoader(file_path)
        docs = loader.load()
    else:
        print(f"  ⚠️ Skipping unsupported file format: {filename}")
        continue

    # Enrich metadata for complete traceability
    for doc in docs:
        doc.metadata["source"] = file_path
        if "page" not in doc.metadata:
            doc.metadata["page"] = 0

    documents.extend(docs)

print(f"\n✅ Loaded {len(documents)} total pages from {len(file_list)} document(s)")
print(f"⏱ Ingestion time: {time.time() - start_time:.2f} seconds")


## Step 6: Document Chunking (Recursive Character Splitting)
Large documents cannot be sent directly into the language model. We divide them into overlapping chunks using `RecursiveCharacterTextSplitter`:
- `chunk_size = 1000`: Balances context richness with retrieval specificity.
- `chunk_overlap = 100`: Preserves continuity of statements that span across chunk boundaries.


In [ ]:
#@title ✂️ Chunking Configuration
chunk_size = 1000 #@param {type:"integer"}
chunk_overlap = 100 #@param {type:"integer"}

from langchain_text_splitters import RecursiveCharacterTextSplitter

print(f"✂️ Splitting documents into chunks (size={chunk_size}, overlap={chunk_overlap})...")
split_start = time.time()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

texts = text_splitter.split_documents(documents)

print(f"✅ Total chunks created: {len(texts)}")
print(f"⏱ Splitting completed in {time.time() - split_start:.2f} seconds")

# Preview the first chunk
if texts:
    first_chunk = texts[0]
    src = os.path.basename(first_chunk.metadata.get('source', ''))
    pg = first_chunk.metadata.get('page', 0) + 1
    print(f"\n🔍 Chunk #1 Preview [Source: {src}, Page: {pg}]:")
    print(first_chunk.page_content[:250].strip() + "...")


## Step 7: Embeddings Generation & ChromaDB Vector Store
We load the `BAAI/bge-small-en` embedding model:
- Runs efficiently on CPU (matching the original tutorial, as `bge-small-en` is lightweight with only 33M parameters).
- Embeddings are normalized (`normalize_embeddings=True`) so that cosine distance is accurately measured.
- The chunks and embeddings are indexed into **ChromaDB** with persistent directory `chroma_db`.


In [ ]:
#@title 🧠 Build / Refresh Chroma Vector Store
REBUILD_VECTOR_DB = True #@param {type:"boolean"}

import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("🧠 Loading 'BAAI/bge-small-en' embedding model on CPU...")

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

CHROMA_PATH = "chroma_db"

if REBUILD_VECTOR_DB and os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print(f"🧹 Cleaned existing vector store at '{CHROMA_PATH}' for fresh indexing.")

if not os.path.exists(CHROMA_PATH):
    print(f"📦 Indexing {len(texts)} chunks into ChromaDB...")
    t0 = time.time()
    vectorstore = Chroma.from_documents(
        documents=texts,
        embedding=embedding_model,
        persist_directory=CHROMA_PATH,
        ids=[str(i) for i in range(len(texts))]
    )
    print(f"✅ ChromaDB built in {time.time() - t0:.2f} seconds!")
else:
    print(f"📂 Loading existing ChromaDB from '{CHROMA_PATH}'...")
    vectorstore = Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=embedding_model
    )

print(f"📊 Vector store indexed chunk count: {vectorstore._collection.count()}")


## Step 8: Semantic Retrieval & Retrieval-Quality Filtering
When a question is queried against the vector database:
1. `vectorstore.similarity_search_with_score(question, k=3)` retrieves the 3 closest chunks.
2. The score returned represents **Cosine Distance**:
   $$\text{Cosine Distance} = 1 - \text{Cosine Similarity}$$
   - **Low distance (e.g. 0.15 - 0.45)**: Strong semantic match.
   - **High distance (e.g. > 0.60)**: Weak match or completely irrelevant question.
3. **Hallucination Prevention**: If **all** top chunks exceed the `threshold` (default $0.60$), the query is rejected immediately with:
   ```
   REF pages: N/A
   The manual does not specify this information!!
   ```


In [ ]:
#@title 🔎 Retriever & Threshold Parameters
TOP_K = 3 #@param {type:"integer"}
DISTANCE_THRESHOLD = 0.60 #@param {type:"slider", min:0.2, max:1.0, step:0.05}

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

print(f"✅ Retriever configured with k={TOP_K}, distance threshold={DISTANCE_THRESHOLD:.2f}")


## Step 9: Prompt Engineering & Local LLM Integration
We formulate a prompt template that strictly grounds the LLM:
- Mandates starting with `REF pages: <page numbers>`
- Restricts responses purely to the supplied context
- Enforces a concise one-paragraph summary (<= 50 words)


In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_ollama import OllamaLLM

ANSWER_PROMPT = PromptTemplate(
    input_variables=["context", "question", "refpages"],
    template="""
REF pages: {refpages}

Context:
{context}

Question:
{question}

Answer in one paragraph (max 50 words).
"""
)

# System instructions: strict grounding rules
system_instructions = (
    "You must start every answer with:\n"
    "REF pages: <comma seperated list>\n"
    "You must answer strictly from the provided context.\n"
    "Do not use outside knowledge.\n\n"
    "If the context does not contain the answer, reply exactly:\n"
    "The manual does not specify this information!!\n"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instructions),
    ("user", "{input}")
])

print(f"🤖 Initializing Ollama model '{MODEL_NAME}'...")
llm = OllamaLLM(model=MODEL_NAME)
chain = prompt | llm
print("✅ LangChain LCEL pipeline built successfully!")


## Step 10: Complete `ask_question()` Pipeline Function
This core function coordinates:
1. Vector similarity search with scores
2. Cosine distance thresholding
3. Context assembly and page reference extraction
4. LLM inference via Ollama


In [ ]:
def ask_question(question: str, k: int = TOP_K, threshold: float = DISTANCE_THRESHOLD, verbose: bool = True):
    """
    Full RAG Pipeline execution:
    1. Retrieves similar chunks + distance scores
    2. Filters matches against distance threshold
    3. Builds context & extracts reference pages
    4. Queries local Llama 3.2 model
    """
    if verbose:
        print(f"🔎 Searching relevant information for: '{question}'\n")

    # Step 1: Retrieve top-k chunks with distance scores
    docs_scores = vectorstore.similarity_search_with_score(question, k=k)

    if verbose:
        print("------------------------------------")
        for doc, score in docs_scores:
            print(f"COSINE DISTANCE SCORE = {score:.3f}")
        print("------------------------------------\n")

    # Step 2: Quality Filtering Threshold Check
    all_bad = all(score > threshold for _, score in docs_scores)
    if all_bad:
        if verbose:
            print(f"⚠️ All retrieved chunks exceeded the distance threshold of {threshold:.2f}!")
            print("🚫 Query rejected to prevent hallucination.\n")
        return "REF pages: N/A\nThe manual does not specify this information!!", docs_scores

    # Step 3: Extract documents and show debug information
    docs = [doc for doc, _ in docs_scores]

    if verbose:
        print("--- Retrieved Chunks ---")
        for i, doc in enumerate(docs):
            src_name = os.path.basename(doc.metadata.get('source', 'Unknown'))
            raw_page = doc.metadata.get('page', -1)
            display_page = raw_page + 1 if raw_page >= 0 else 'N/A'
            snippet = doc.page_content.replace("\n", " ").strip()
            print(f"[{i+1}] Source: {src_name} | Page: {display_page}")
            print(f"    {snippet[:140]}...")
        print("------------------------\n")

    # Step 4: Build context from retrieved chunks
    context = "\n\n".join(doc.page_content for doc in docs)

    # Step 5: Extract 1-indexed reference page numbers
    refpages = sorted({
        str(doc.metadata.get("page", -1) + 1)
        for doc in docs
        if doc.metadata.get("page", -1) >= 0
    })
    ref_string = ", ".join(refpages) if refpages else "N/A"

    # Step 6: Format prompt
    filled_prompt = ANSWER_PROMPT.format(
        context=context,
        question=question,
        refpages=ref_string
    )

    # Step 7: Invoke LLM chain
    t0 = time.time()
    response = chain.invoke({"input": filled_prompt})
    elapsed = time.time() - t0

    if verbose:
        print(f"⏱ LLM response generated in {elapsed:.2f} seconds\n")

    return response, docs_scores

print("✅ 'ask_question' pipeline function registered!")


## Step 11: Interactive User Interface
We provide three ways to interact with the RAG system:
1. **Interactive Dashboard (`ipywidgets`)**: A graphical user interface with query presets, score badges, and response viewer.
2. **Colab Form Field**: Single query field right in the notebook cell.
3. **Continuous CLI Loop**: Terminal-style chat session.


In [ ]:
#@title 💻 Interactive RAG Dashboard (ipywidgets)
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

query_box = widgets.Text(
    value="What first aid should be given if acetone gets into the eyes?",
    placeholder="Type your question about the uploaded documents here...",
    description="Question:",
    layout=widgets.Layout(width="75%")
)

ask_btn = widgets.Button(
    description="Ask LLM",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="120px")
)

clear_btn = widgets.Button(
    description="Clear",
    button_style="",
    icon="trash",
    layout=widgets.Layout(width="80px")
)

sample_dropdown = widgets.Dropdown(
    options=[
        ("Select a sample question...", ""),
        ("Acetone: First aid for eye contact", "What first aid should be given if acetone gets into the eyes?"),
        ("Acetone: Flash point & flammability", "What is the flash point and flammability hazard of acetone?"),
        ("NaOH: Personal Protective Equipment (PPE)", "What personal protective equipment is required when handling sodium hydroxide?"),
        ("NaOH: Incompatible materials", "Which materials or chemicals are incompatible with sodium hydroxide?"),
        ("Rejection Test (Irrelevant query)", "Who won the FIFA World Cup in 2022?")
    ],
    value="",
    description="Presets:",
    layout=widgets.Layout(width="75%")
)

def on_sample_change(change):
    if change['new']:
        query_box.value = change['new']

sample_dropdown.observe(on_sample_change, names='value')

output_area = widgets.Output()

def run_query(b):
    q = query_box.value.strip()
    if not q:
        return
    with output_area:
        clear_output()
        print("🤖 Searching vector database and querying local LLM...")
        t0 = time.time()
        ans, scores = ask_question(q, verbose=False)
        total_time = time.time() - t0
        clear_output()

        # Score badges
        score_badges = ""
        for i, (doc, sc) in enumerate(scores):
            color = "#28a745" if sc <= DISTANCE_THRESHOLD else "#dc3545"
            src_name = os.path.basename(doc.metadata.get('source', 'doc'))
            pg = doc.metadata.get('page', -1) + 1
            score_badges += f"<span style='background-color:{color}; color:white; padding:3px 8px; border-radius:12px; margin-right:6px; font-size:12px;'>Chunk {i+1} ({src_name} p.{pg}): {sc:.3f}</span>"

        html_card = f"""
        <div style="border: 1px solid #e0e0e0; border-radius: 8px; padding: 16px; background-color: #fdfdfd; margin-top: 10px; font-family: sans-serif;">
            <div style="font-size: 14px; color: #555; margin-bottom: 6px;"><strong>Question:</strong> {q}</div>
            <div style="margin-bottom: 12px;"><strong>Cosine Distance Scores:</strong><br>{score_badges}</div>
            <div style="padding: 12px; background-color: #f0f4f8; border-left: 4px solid #1a73e8; border-radius: 4px; font-size: 15px; line-height: 1.5; color: #222;">
                <strong>Response:</strong><br>{ans.replace('\n', '<br>')}
            </div>
            <div style="font-size: 11px; color: #888; margin-top: 8px;">⏱ Total response time: {total_time:.2f} seconds</div>
        </div>
        """
        display(HTML(html_card))

def clear_all(b):
    query_box.value = ""
    with output_area:
        clear_output()

ask_btn.on_click(run_query)
clear_btn.on_click(clear_all)

ui = widgets.VBox([
    sample_dropdown,
    widgets.HBox([query_box, ask_btn, clear_btn]),
    output_area
])

display(ui)


In [ ]:
#@title 💬 Quick Single Question Form
#@markdown Type your question below and run this cell for a single query:
question_input = "What are the first aid measures if acetone gets into eyes?" #@param {type:"string"}
distance_cutoff = 0.60 #@param {type:"number"}

answer, docs_scores = ask_question(question_input, threshold=distance_cutoff, verbose=True)

print("=" * 50)
print(f"❓ QUESTION:\n{question_input}\n")
print(f"✅ ANSWER:\n{answer}")
print("=" * 50)


In [ ]:
#@title 🔄 Interactive Chat Loop (Terminal Style)
#@markdown Run this cell to start an interactive console session. Type 'exit' or 'quit' to end.

print("📚 RAG Interactive Session Active")
print("Type 'exit' or 'quit' to end session.\n")

while True:
    try:
        user_q = input("💬 Ask a question:\n> ")
    except EOFError:
        break

    if user_q.lower().strip() in ["exit", "quit", "q"]:
        print("👋 Session ended.")
        break

    if not user_q.strip():
        continue

    t0 = time.time()
    resp, _ = ask_question(user_q, verbose=False)
    elapsed = time.time() - t0

    print("\n" + "=" * 45)
    print(f"❓ Question: {user_q}")
    print(f"✅ Response:\n{resp}")
    print(f"⏱ Time: {elapsed:.2f}s")
    print("=" * 45 + "\n")


## Step 12: Summary & Key Takeaways
1. **Interactive File Ingestion**: Users can upload any custom Safety Data Sheets (or manuals) directly without requiring Google Drive mounting.
2. **Metadata Tracking**: Preserving document source paths and page numbers enables complete auditability and source verification.
3. **Retrieval Quality Filtering**: Setting a cosine distance threshold ($0.60$) prevents language model hallucinations when users ask questions beyond the scope of the documents.
4. **Local LLM with Ollama**: Running Llama 3.2 via Ollama locally inside Google Colab provides complete data privacy, eliminates API costs, and runs efficiently on both CPU and T4 GPU runtimes.
